# Import libraries

In [ ]:
import pandas as pd
import numpy as np 
from transformers import AutoTokenizer, AutoModel

import ollama
import psycopg2
from pgvector.psycopg2 import register_vector
import torch

# 20NG Import

In [ ]:
df_20NG                 = pd.read_csv('20260211_20NG_Dataset.csv')
df_20NG_subject         = df_20NG['Subject']
df_20NG_from            = df_20NG['From']
df_20NG_organization    = df_20NG['Organization']
df_20NG_newsgroup       = df_20NG['Newsgroup']
df_20NG_body            = df_20NG['Body']

# two groups 
subject_body_20NG       = df_20NG_subject.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)
from_body_20NG          = df_20NG_from.reset_index(drop=True)           + ' ' + df_20NG_body.reset_index(drop=True)
organisation_body_20NG  = df_20NG_organization.reset_index(drop=True)   + ' ' + df_20NG_body.reset_index(drop=True)
newsgroup_body_20NG     = df_20NG_newsgroup.reset_index(drop=True)      + ' ' + df_20NG_body.reset_index(drop=True)

# three groups 
subject_from_body_20NG              = df_20NG_subject.reset_index(drop=True)        + ' ' + df_20NG_from.reset_index(drop=True)             + ' ' + df_20NG_body.reset_index(drop=True)
subject_organization_body_20NG      = df_20NG_subject.reset_index(drop=True)        + ' ' + df_20NG_organization.reset_index(drop=True)     + ' ' + df_20NG_body.reset_index(drop=True)
subject_newsgroup_body_20NG         = df_20NG_subject.reset_index(drop=True)        + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)
from_organization_body_20NG         = df_20NG_from.reset_index(drop=True)           + ' ' + df_20NG_organization.reset_index(drop=True)     + ' ' + df_20NG_body.reset_index(drop=True)
from_newsgroup_body_20NG            = df_20NG_from.reset_index(drop=True)           + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)
organization_newsgroup_body_20NG    = df_20NG_organization.reset_index(drop=True)   + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)

# four groups 
subject_from_organization_body_20NG         = df_20NG_subject.reset_index(drop=True) + ' ' + df_20NG_from.reset_index(drop=True)            + ' ' + df_20NG_organization.reset_index(drop=True)     + ' ' + df_20NG_body.reset_index(drop=True)
subject_from_newsgroup_body_20NG            = df_20NG_subject.reset_index(drop=True) + ' ' + df_20NG_from.reset_index(drop=True)            + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)
subject_organization_newsgroup_body_20NG    = df_20NG_subject.reset_index(drop=True) + ' ' + df_20NG_organization.reset_index(drop=True)    + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)
from_organization_newsgroup_body_20NG       = df_20NG_from.reset_index(drop=True)    + ' ' + df_20NG_organization.reset_index(drop=True)    + ' ' + df_20NG_newsgroup.reset_index(drop=True)        + ' ' + df_20NG_body.reset_index(drop=True)

# five groups 
subject_from_organization_newsgroup_body_20NG    = df_20NG_subject.reset_index(drop=True) + ' ' + df_20NG_from.reset_index(drop=True)            + ' ' + df_20NG_organization.reset_index(drop=True)     + ' ' + df_20NG_newsgroup.reset_index(drop=True) + ' ' + df_20NG_body.reset_index(drop=True)

## 20NG unique values

In [ ]:
# 9441 unique 'Subject'
unique_subject = df_20NG_subject.unique()
print(unique_subject)
print(len(unique_subject))

In [ ]:
# 7832 unique 'From'
unique_total = df_20NG_from.unique()
print(unique_total)
print(len(unique_total))

In [ ]:
# 3873 unique 'Organization'
unique_total = df_20NG_organization.unique()
print(unique_total)
print(len(unique_total))

In [ ]:
# 20 unique 'Newsgroups'
unique_total = df_20NG_newsgroup.unique()
print(unique_total)
print(len(unique_total))

In [ ]:
# 16739 unique 'Body'
unique_total = df_20NG_body.unique()
print(unique_total)
print(len(unique_total))

## Check missing data 

In [ ]:
def check_empty_data(df_data): 

    data = df_data.columns.tolist()

    for i in range(len(data)): 

        data_val = data[i]
        empty_value = df_20NG[data_val].isna().sum()

        print(f"Number of empty {data_val} rows: {empty_value}")

In [ ]:
check_empty_data(df_20NG)

## Token length

In [ ]:
def calc_aver_token_size(text_data_df, model):

    tokeniser   = AutoTokenizer.from_pretrained(model)
    tkn_arr     = []

    for i in range(len(text_data_df)):

        text = str(text_data_df[i])
        input_ids = tokeniser(text)["input_ids"]
        num_tokens = len(input_ids)
        tkn_arr.append(num_tokens)

    return tkn_arr


def calc_token_ranges(text, model_arr, text_name): 

    for i in range(len(model_arr)): 

        # return the tkn_arr - num_tokens - from calc_aver_token_size
        tkn_arr         = calc_aver_token_size(text, model_arr[i])

        # find the average token length for all the records
        tkn_len         = len(tkn_arr)
        aver_tkn_len    = sum(tkn_arr)/tkn_len
        max_tkn_len     = max(tkn_arr)

        # find the token ranges for the text, up to 8192 tokens
        tkn_arr         = np.array(tkn_arr)

        # from 0 to the max
        zero_128_range      = np.sum((tkn_arr >= 0) & (tkn_arr <= 128))
        perc_0_128          = 100 * zero_128_range / tkn_len

        zero_256_range      = np.sum((tkn_arr >= 0) & (tkn_arr <= 256))
        perc_0_256          = 100 * zero_256_range / tkn_len

        zero_512_range      = np.sum((tkn_arr >= 0) & (tkn_arr <= 512))
        perc_0_512          = 100 * zero_512_range / tkn_len
        
        zero_1024_range     = np.sum((tkn_arr >= 0) & (tkn_arr <= 1024))
        perc_0_1024         = 100 * zero_1024_range / tkn_len

        zero_2048_range     = np.sum((tkn_arr >= 0) & (tkn_arr <= 2048))
        perc_0_2048         = 100 * zero_2048_range / tkn_len

        zero_4096_range     = np.sum((tkn_arr >= 0) & (tkn_arr <= 4096))
        perc_0_4096         = 100 * zero_4096_range / tkn_len

        zero_8192_range     = np.sum((tkn_arr >= 0) & (tkn_arr <= 8192))
        perc_0_8192         = 100 * zero_8192_range / tkn_len

        # Print values
        print(" ")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
        print(f"Chosen text                 ; {text_name}")
        print(f"Model name                  : {model_arr[i]}")
        print(f"Average token length        : {aver_tkn_len}")
        print(f"Maximum token length        : {max_tkn_len}")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
        print(f"0-128 token percentage      : {perc_0_128}")
        print(f"0-256 token percentage      : {perc_0_256}")
        print(f"0-512 token percentage      : {perc_0_512}")
        print(f"0-1024 token percentage     : {perc_0_1024}")
        print(f"0-2048 token percentage     : {perc_0_2048}")
        print(f"0-4096 token percentage     : {perc_0_4096}")
        print(f"0-8192 token percentage     : {perc_0_8192}")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")


def calc_intermediate_token_ranges(text, model_arr, text_name): 

    for i in range(len(model_arr)): 

        # return the tkn_arr - num_tokens - from calc_aver_token_size
        tkn_arr         = calc_aver_token_size(text, model_arr[i])

        # find the average token length for all the records
        tkn_len         = len(tkn_arr)
        aver_tkn_len    = sum(tkn_arr)/tkn_len
        max_tkn_len     = max(tkn_arr)

        # find the token ranges for the text, up to 8192 tokens
        tkn_arr         = np.array(tkn_arr)

        # from 0 to the max
        zero_128_range      = np.sum((tkn_arr >= 0) & (tkn_arr <= 128))
        perc_0_128          = 100 * zero_128_range / tkn_len

        range_129_256       = np.sum((tkn_arr >= 129) & (tkn_arr <= 256))
        perc_129_256        = 100 * range_129_256 / tkn_len

        range_257_512       = np.sum((tkn_arr >= 257) & (tkn_arr <= 512))
        perc_257_512        = 100 * range_257_512 / tkn_len
        
        range_513_1024      = np.sum((tkn_arr >= 513) & (tkn_arr <= 1024))
        perc_513_1024       = 100 * range_513_1024 / tkn_len

        range_1025_2048     = np.sum((tkn_arr >= 1025) & (tkn_arr <= 2048))
        perc_1025_2048      = 100 * range_1025_2048 / tkn_len

        range_2049_4096     = np.sum((tkn_arr >= 2049) & (tkn_arr <= 4096))
        perc_2049_4096      = 100 * range_2049_4096 / tkn_len

        range_4097_8192     = np.sum((tkn_arr >= 4097) & (tkn_arr <= 8192))
        perc_4087_8192      = 100 * range_4097_8192 / tkn_len

        # Print values
        print(" ")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
        print(f"Chosen text                 ; {text_name}")
        print(f"Model name                  : {model_arr[i]}")
        print(f"Average token length        : {aver_tkn_len}")
        print(f"Maximum token length        : {max_tkn_len}")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
        print(f"0-128 token percentage      : {perc_0_128}")
        print(f"129-256 token percentage    : {perc_129_256}")
        print(f"257-512 token percentage    : {perc_257_512}")
        print(f"513-1024 token percentage   : {perc_513_1024}")
        print(f"1025-2048 token percentage  : {perc_1025_2048}")
        print(f"2049-4096 token percentage  : {perc_2049_4096}")
        print(f"4097-8192 token percentage  : {perc_4087_8192}")
        print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%") 

In [ ]:
# Model name
EM_arr      = ["./models/allminilm", "./models/bert", "./models/bigbird", "./models/e5", "./models/longformer", "./models/qwen3"]

### 20NG - body (only)

In [ ]:
text_name   = "body (only)" 
df_text     = df_20NG_body

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + body

In [ ]:
text_name   = "subject + body" 
df_text     = subject_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - from + body

In [ ]:
text_name   = "from + body" 
df_text     = from_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - organization + body

In [ ]:
text_name   = "organization + body" 
df_text     = organisation_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - newsgroup + body

In [ ]:
text_name   = "newsgroup + body" 
df_text     = newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + from + body

In [ ]:
text_name   = "subject + from + body" 
df_text     = subject_from_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + organization + body

In [ ]:
text_name   = "subject + organization + body" 
df_text     = subject_organization_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + newsgroup + body

In [ ]:
text_name   = "subject + newsgroup + body" 
df_text     = subject_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - from + organization + body

In [ ]:
text_name   = "from + organization + body" 
df_text     = from_organization_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - from + newsgroup + body

In [ ]:
text_name   = "from + newsgroup + body" 
df_text     = from_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - organization + newsgroup + body

In [ ]:
text_name   = "organization + newsgroup + body" 
df_text     = organization_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + from + organization + body

In [ ]:
text_name   = "subject + from + organization + body" 
df_text     = subject_from_organization_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + from + newsgroup + body

In [ ]:
text_name   = "subject + from + newsgroup + body" 
df_text     = subject_from_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + organization + newsgroup + body

In [ ]:
text_name   = "subject + organization + newsgroup + body" 
df_text     = subject_organization_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - from + organization + newsgroup + body

In [ ]:
text_name   = "from + organization + newsgroup + body" 
df_text     = from_organization_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)

### 20NG - subject + from + organization + newsgroup + body

In [ ]:
text_name   = "subject + from + organization + newsgroup + body" 
df_text     = subject_from_organization_newsgroup_body_20NG

In [ ]:
calc_token_ranges(df_text, EM_arr, text_name)

In [ ]:
calc_intermediate_token_ranges(df_text, EM_arr, text_name)